In [ ]:
!python -m pip install --upgrade pip
!pip install -U "transformers==4.49.0"
!pip install -U opencv-python-headless==4.8.1.78
!pip install timm
!pip install -U bitsandbytes
!pip install --user --ignore-installed -U ipywidgets jupyterlab_widgets 

In [ ]:
# ============================================================
# 실습 준비 — 모델·데이터셋 미리 내려받기
# 강의 시작 시 이 셀을 먼저 실행하세요
# ============================================================

# ---------- 0. Hugging Face 토큰 ----------
# 아래 따옴표 안에 토큰을 붙여넣으세요 (없으면 비워두세요)
# 발급: huggingface.co → 프로필 → Settings → Access Tokens → New token (Read)

HF_TOKEN = ""

# ------------------------------------------------------------
print("셀 시작", flush=True)

import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

print("라이브러리 불러오는 중...", flush=True)

import time, random
import datasets
from huggingface_hub import snapshot_download, scan_cache_dir
from datasets import load_dataset

print("완료\n", flush=True)


# ==========================================================
# 토큰 설정
# ==========================================================
token = HF_TOKEN.strip() or os.environ.get("HF_TOKEN", "")

if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    try:
        from huggingface_hub import HfApi
        print("로그인 —", HfApi().whoami()["name"], flush=True)
    except Exception as e:
        print("토큰 확인 실패 —", type(e).__name__, flush=True)
else:
    os.environ.pop("HF_TOKEN", None)
    os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
    print("익명으로 진행 — 요청 제한(429)에 걸리기 쉽습니다", flush=True)

print("캐시 경로 —", datasets.config.HF_DATASETS_CACHE, "\n", flush=True)


# ==========================================================
# 목록
# ==========================================================
MODELS = [
    ("crangana/railroad-fault-detector",             "① 분류"),
    ("theodullin/detr-resnet-50_finetuned_blood_cell_15epochs", "② 탐지"),
    ("facebook/sam-vit-base",                        "③ 분할"),
    ("Intel/dpt-hybrid-midas",                       "④ 깊이 추정"),
    ("nlpconnect/vit-gpt2-image-captioning",         "⑤ 이미지 설명"),
    ("naver-clova-ix/donut-base-finetuned-docvqa",   "⑥ 문서 질의응답"),
    ("flaviagiammarino/git-base-vqarad",             "⑦ 시각 질의응답"),
    ("Salesforce/blip-vqa-base",                     "⑦ 시각 질의응답"),
    ("openai/clip-vit-base-patch32",                 "제로샷"),
    ("google/vit-base-patch16-224-in21k",            "미세조정 백본"),
]

# (이름, split, 설정(config), 설명)
DATASETS = [
    ("crangana/railroad-fault-detection",           "train[:100]",  None,   "① 분류"),
    ("keremberke/blood-cell-object-detection",      "train[:100]",  "full", "② 탐지"),
    ("keremberke/pcb-defect-segmentation",          "train[:50]",   "full", "③ 분할"),
    ("merve/scene_parse_150",                       "train[:50]",   None,   "④⑤ 깊이·설명"),
    ("Anas989898/Vision-OCR-Financial-Reports-10k", "train[:50]",   None,   "⑥ 문서 QA"),
    ("Kaith-jeet123/brain_tumor_vqa",               "train[:50]",   None,   "⑦ 시각 QA"),
    ("imageomics/IDLE-OO-Camera-Traps",             "test[:100]",   None,   "제로샷"),
    ("ethz/food101",                                "train[:5000]", None,   "미세조정"),
]


# ==========================================================
# 캐시 확인
# ==========================================================
def cached_model_repos():
    """이미 받아둔 모델 저장소 목록"""
    try:
        return {r.repo_id for r in scan_cache_dir().repos if r.repo_type == "model"}
    except Exception:
        return set()


def dataset_cached(name, split, cfg=None):
    """캐시만으로 불러와지는지 확인 (네트워크 미사용)

    환경변수는 import 시점에만 읽히므로 datasets.config 를 직접 변경한다.
    """
    prev = getattr(datasets.config, "HF_DATASETS_OFFLINE", False)
    datasets.config.HF_DATASETS_OFFLINE = True
    try:
        kw = {"trust_remote_code": True}
        if cfg:
            kw["name"] = cfg
        load_dataset(name, split=split, **kw)
        return True
    except Exception:
        return False
    finally:
        datasets.config.HF_DATASETS_OFFLINE = prev


# ==========================================================
# 재시도
# ==========================================================
# 재시도해도 소용없는 오류 — 코드·설정 문제이므로 즉시 중단
NO_RETRY = (ValueError, KeyError, FileNotFoundError,
            TypeError, ImportError, NotImplementedError)


def retry(fn, name, tries=6):
    for i in range(tries):
        try:
            t0 = time.time()
            fn()
            print(f"  완료 — {time.time() - t0:.0f}초", flush=True)
            ok.append(name)
            return True

        except Exception as e:
            # 1) 설정·코드 오류는 재시도 생략
            if isinstance(e, NO_RETRY):
                print(f"  중단 — 재시도 불가 ({type(e).__name__})", flush=True)
                print(f"    {str(e)[:300]}", flush=True)